# 🤖 Notebook 2: Classical ML Benchmark

Trains and evaluates SVM, Random Forest, XGBoost, and k-NN classifiers
on hand-crafted micro-Doppler features. Includes SHAP explainability.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.preprocessing import label_binarize
import time

from Dataset.loader import load_dataset, get_iq_matrix, get_labels, get_train_test_split
from Preprocessing.features import extract_features, FEATURE_NAMES
from Preprocessing.normalize import fit_feature_scaler
from Classical_ML.train import train_svm, train_random_forest, train_knn
from Evaluation.metrics import compute_metrics, print_metrics

plt.style.use('dark_background')
COLORS = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff']
CLASS_NAMES = ['2-blade', '3-blade', '4-blade']
LABEL_ORDER = [2, 3, 4]
print("✅ Ready")

## 1. Load & Prepare Data

In [ ]:
print("Loading dataset (5000 rows)...")
df = load_dataset('../helicopter_microdoppler_dataset.csv', nrows=5000)
X_iq_train, X_iq_test, y_train, y_test = get_train_test_split(df, test_size=0.2)

print("Extracting features...")
X_train_raw = extract_features(X_iq_train)
X_test_raw  = extract_features(X_iq_test)

scaler = fit_feature_scaler(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")

## 2. Train All Classifiers

In [ ]:
models = {}
timings = {}

configs = [
    ('SVM (RBF, C=10)',  train_svm,           {'C': 10.0}),
    ('Random Forest',    train_random_forest, {'n_estimators': 200}),
    ('k-NN (k=7)',       train_knn,           {'n_neighbors': 7}),
]

for name, fn, kwargs in configs:
    print(f"Training {name}...")
    t0 = time.time()
    models[name] = fn(X_train, y_train, **kwargs)
    timings[name] = time.time() - t0
    print(f"  Done in {timings[name]:.1f}s")

## 3. Results & Comparison

In [ ]:
all_metrics = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
    all_metrics[name] = compute_metrics(y_test, y_pred, y_proba)

# Summary table
print(f"\n{'Model':<25} {'Accuracy':>10} {'Macro-F1':>10} {'ROC-AUC':>10} {'Time':>8}")
print("─" * 68)
for name, m in all_metrics.items():
    auc  = f"{m['roc_auc']:.4f}" if m.get('roc_auc') else "  N/A  "
    t    = f"{timings[name]:.1f}s"
    print(f"{name:<25} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f} {auc:>10} {t:>8}")

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices — Classical ML Models', fontsize=15, fontweight='bold')

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, y_pred, labels=LABEL_ORDER)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = all_metrics[name]['accuracy']
    ax.set_title(f'{name}\nAccuracy: {acc:.3f}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices_classical.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(
    [FEATURE_NAMES[i] for i in sorted_idx],
    importances[sorted_idx],
    color=COLORS[:len(FEATURE_NAMES)] * 5,
    edgecolor='white', linewidth=0.3, alpha=0.9
)
ax.set_title('Random Forest Feature Importances\n(Gini Impurity Decrease)', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance')
ax.set_xlabel('Feature')
plt.xticks(rotation=45, ha='right')
ax.grid(True, alpha=0.2, axis='y')

# Highlight top feature
top_feat = FEATURE_NAMES[sorted_idx[0]]
print(f"\n🏆 Most important feature: '{top_feat}' ({importances[sorted_idx[0]]:.4f})")
print(f"   This corresponds to the blade flash rate — a direct physical signature!")

plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 6. SNR Robustness

In [ ]:
from Robustness_Testing.noise_robustness import snr_sweep, plot_snr_curve

svm_model = models['SVM (RBF, C=10)']
rf_model_  = models['Random Forest']

results_dict = {}
for name, mdl in [('SVM', svm_model), ('Random Forest', rf_model_)]:
    print(f"\nSNR sweep — {name}")
    def predict_fn(X_iq_noisy, m=mdl):
        feat = extract_features(X_iq_noisy)
        return m.predict(scaler.transform(feat))
    results_dict[name] = snr_sweep(predict_fn, X_iq_test[:300], y_test[:300], snr_range=range(-5, 31, 5))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, snr_res) in enumerate(results_dict.items()):
    snrs = sorted(snr_res.keys())
    accs = [snr_res[s] for s in snrs]
    ax.plot(snrs, accs, 'o-', linewidth=2.5, markersize=7, color=COLORS[i], label=name)

ax.axhline(y=1/3, color='white', linestyle='--', alpha=0.4, label='Random baseline')
ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Robustness: Accuracy vs SNR', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('snr_robustness_classical.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Classical ML benchmark complete!")